# Pose and wheeled-robot motion: notebook

The differential-drive robot of the Note: wheel radius 0.05 m, wheel separation 0.2 m. We check the Note's numbers, predict paths with Euler steps, and try the controls ourselves.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go

# The robot's two measurements (metres)
r = 0.05   # wheel radius
L = 0.2    # distance between the two wheels

## 1. Wheel speeds to robot speeds, and back

Forward kinematics: forward speed = average of the wheel ground speeds; turn rate = their difference divided by L. Inverse kinematics turns a wanted (v, ω) into wheel spin rates.

In [2]:
def forward(uR, uL):
    """Wheel spin rates (rad/s) -> forward speed v (m/s), turn rate omega (rad/s)."""
    vR, vL = r * uR, r * uL            # rolling without slipping: ground speed = radius x spin rate
    return (vR + vL) / 2, (vR - vL) / L

def inverse(v, omega):
    """Wanted v (m/s), omega (rad/s) -> wheel spin rates (rad/s)."""
    vR = v + omega * L / 2
    vL = v - omega * L / 2
    return vR / r, vL / r

print(forward(12, 8))     # Note, Section 4.3: (0.5, 1.0)
print(inverse(0.3, 0.5))  # Note, Section 4.5: (7.0, 5.0)

(0.5, 1.0000000000000002)
(6.999999999999999, 5.0)


## 2. The four ways to drive (Note, Section 4.4)

In [3]:
rows = []
for vR, vL in [(0.5, 0.5), (0.2, -0.2), (0.4, 0.0), (0.6, 0.4)]:
    v, w = forward(vR / r, vL / r)
    R = v / w if abs(w) > 1e-12 else np.inf
    rows.append(dict(vR=vR, vL=vL, v=round(v, 3), omega=round(w, 3), R=R))
pd.DataFrame(rows)

,vR,vL,v,omega,R
0,0.5,0.5,0.5,0.0,inf
1,0.2,-0.2,0.0,2.0,0.0
2,0.4,0.0,0.2,2.0,0.1
3,0.6,0.4,0.5,1.0,0.5


## 3. Predicting the path with Euler steps (Note, Section 5.3)

In [4]:
def step(q, v, w, dt):
    """One Euler step of the kinematic model x' = v cos(th), y' = v sin(th), th' = w."""
    x, y, th = q
    return (x + v * np.cos(th) * dt, y + v * np.sin(th) * dt, th + w * dt)

q = (2.0, 1.0, np.radians(30))   # pose of Figure 1
table = []
for k in range(3):
    x, y, th = q
    table.append(dict(step=k, x=round(x, 4), y=round(y, 4), theta=round(th, 4),
                      x_rate=round(0.5 * np.cos(th), 4), y_rate=round(0.5 * np.sin(th), 4)))
    q = step(q, 0.5, 1.0, 0.1)
pd.DataFrame(table)

,step,x,y,theta,x_rate,y_rate
0,0,2.0000,1.0000,0.5236,0.4330,0.250
1,1,2.0433,1.0250,0.6236,0.4059,0.292
2,2,2.0839,1.0542,0.7236,0.3747,0.331


The step size matters. We compare each Euler path with the exact circle (centre 0.5 m to the robot's left, radius v/ω = 0.5 m) at the same times, over one lap.

In [5]:
def largest_gap(dt, v=0.5, w=1.0):
    th0 = np.radians(30)
    q, gap = (2.0, 1.0, th0), 0.0
    for k in range(1, round(2 * np.pi / dt) + 1):
        q = step(q, v, w, dt)
        t = k * dt
        xe = 2 + (v / w) * (np.sin(th0 + w * t) - np.sin(th0))      # exact position on the circle at time t
        ye = 1 - (v / w) * (np.cos(th0 + w * t) - np.cos(th0))
        gap = max(gap, np.hypot(q[0] - xe, q[1] - ye))
    return gap

pd.DataFrame({"dt (s)": [1, 0.5, 0.1, 0.01],
              "largest gap (cm)": [round(100 * largest_gap(dt), 1) for dt in (1, 0.5, 0.1, 0.01)]})

,dt (s),largest gap (cm)
0,1.00,50.6
1,0.50,25.0
2,0.10,5.0
3,0.01,0.5


## 4. Try it: change the right wheel's speed

The slider sets the right wheel's ground speed; the left wheel stays at 0.3 m/s. Each path runs for 4 s from the origin, facing along x.

In [6]:
def path(vR, vL, T=4.0, dt=0.01):
    v, w = (vR + vL) / 2, (vR - vL) / L
    q, pts = (0.0, 0.0, 0.0), [(0.0, 0.0)]
    for _ in range(round(T / dt)):
        q = step(q, v, w, dt)
        pts.append(q[:2])
    return np.array(pts)

speeds = np.round(np.arange(-0.3, 0.61, 0.1), 2)
fig = go.Figure()
for i, vR in enumerate(speeds):
    p = path(vR, 0.3)
    fig.add_trace(go.Scatter(x=p[:, 0], y=p[:, 1], mode="lines", visible=(i == len(speeds) - 1),
                             line=dict(width=4), name=f"vR = {vR}"))
steps = [dict(method="update", label=f"{vR}", args=[{"visible": [j == i for j in range(len(speeds))]}])
         for i, vR in enumerate(speeds)]
fig.update_layout(sliders=[dict(active=len(speeds) - 1, steps=steps, currentvalue=dict(prefix="right wheel (m/s): "))],
                  xaxis=dict(range=[-1.5, 2.5], title="x (m)"), yaxis=dict(range=[-1.5, 2], title="y (m)", scaleanchor="x"),
                  template="simple_white", width=700, height=650, showlegend=False)
fig.show()

## 5. The no-sideways rule (Note, Section 6.1)

The velocity's part along the robot's left direction must be zero: −ẋ sin θ + ẏ cos θ = 0.

In [7]:
def sideways_part(xdot, ydot, th):
    return -xdot * np.sin(th) + ydot * np.cos(th)

th = np.radians(30)
print("drive forward:", round(sideways_part(0.5 * np.cos(th), 0.5 * np.sin(th), th), 6))   # 0: allowed
print("slide left:   ", round(sideways_part(-0.25, 0.433, th), 3))                          # 0.5: forbidden
G = np.array([[np.cos(th), 0], [np.sin(th), 0], [0, 1]])   # columns: drive forward, turn
A = np.array([[-np.sin(th), np.cos(th), 0]])
print("A G =", np.round(A @ G, 12))                          # both allowed directions pass the rule

drive forward: 0.0
slide left:    0.5
A G = [[0. 0.]]


## 6. A trailer, forward and reverse (Note, Section 8)

Trailer heading rate: θ₁' = (v / d₁) sin(θ₀ − θ₁), with d₁ = 2 m. The car drives straight at heading 40°; the trailer starts at 10°.

In [8]:
def angle_between(v, T, d1=2.0, dt=0.01):
    th0, th1, out = np.radians(40), np.radians(10), []
    for k in range(round(T / dt) + 1):
        out.append((k * dt, np.degrees(th0 - th1)))
        th1 += v / d1 * np.sin(th0 - th1) * dt
    return np.array(out)

print("rate at the start:", 1 / 2 * np.sin(np.radians(30)), "rad/s")   # 0.25
fig = go.Figure()
for v, name in [(1.0, "forward, v = +1 m/s"), (-1.0, "reverse, v = -1 m/s")]:
    a = angle_between(v, 6.0 if v > 0 else 2.6)
    fig.add_trace(go.Scatter(x=a[:, 0], y=a[:, 1], mode="lines", name=name, line=dict(width=4)))
fig.update_layout(xaxis_title="time (s)", yaxis_title="angle between car and trailer (degrees)",
                  template="simple_white", width=750, height=450)
fig.show()

rate at the start: 0.24999999999999997 rad/s
